# 01 - Data Understanding

Customer Churn Analysis — Telecom/Subscription Business

This notebook explores the raw dataset before any cleaning: shape, types, missing values, duplicates, and the target distribution.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
df = pd.read_csv('../dataset/customer_churn.csv')
df.shape

(7525, 27)

## Dataset shape and first look

In [2]:
df.head()

,CustomerID,Gender,Age,SeniorCitizen,MaritalStatus,Dependents,Tenure,ContractType,InternetService,PhoneService,StreamingTV,StreamingMovies,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,MonthlyCharges,TotalCharges,PaymentMethod,PaperlessBilling,NumberOfComplaints,CustomerSatisfactionScore,SupportTickets,LastInteractionDate,ReferralStatus,CustomerLifetimeValue,Churn
0,CUST-100000,Female,40,0,Married,Yes,22,Month-to-Month,Fiber Optic,No,No,No,No,Yes,No,No,83.12,1832.84,Mailed Check,Yes,1,3,2,2025-10-16,No,2724.64,Yes
1,CUST-100001,Male,23,0,Married,Yes,43,Month-to-Month,No,No,No Internet Service,No Internet Service,No Internet Service,No Internet Service,No Internet Service,No Internet Service,23.70,995.73,Credit Card (Auto),No,1,4,2,2025-06-29,Yes,1213.14,No
2,CUST-100002,Female,38,0,Widowed,No,28,Month-to-Month,DSL,Yes,No,Yes,No,No,No,No,72.93,1965.35,Bank Transfer (Auto),Yes,0,3,2,2026-01-22,No,3242.81,No
3,CUST-100003,Female,27,0,Married,No,1,Month-to-Month,Fiber Optic,Yes,No,No,No,Yes,No,No,93.85,94.91,Bank Transfer (Auto),No,1,3,4,2026-04-14,Yes,1697.67,Yes
4,CUST-100004,Male,37,0,Single,Yes,0,Month-to-Month,No,Yes,No Internet Service,No Internet Service,No Internet Service,No Internet Service,No Internet Service,No Internet Service,25.77,0.00,Credit Card (Auto),Yes,2,3,5,2025-09-27,No,430.29,No


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7525 entries, 0 to 7524
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CustomerID                 7525 non-null   str    
 1   Gender                     7525 non-null   str    
 2   Age                        7525 non-null   int64  
 3   SeniorCitizen              7525 non-null   int64  
 4   MaritalStatus              7525 non-null   str    
 5   Dependents                 7525 non-null   str    
 6   Tenure                     7525 non-null   int64  
 7   ContractType               7525 non-null   str    
 8   InternetService            7525 non-null   str    
 9   PhoneService               7525 non-null   str    
 10  StreamingTV                7525 non-null   str    
 11  StreamingMovies            7525 non-null   str    
 12  OnlineSecurity             7525 non-null   str    
 13  OnlineBackup               7525 non-null   str    
 14  Dev

## Missing values

In [4]:
missing = df.isna().sum()
missing[missing > 0]

TotalCharges    37
dtype: int64

**Observation:** `TotalCharges` has missing values — these are new customers whose first bill hasn't posted, or export gaps. Will be imputed using `Tenure x MonthlyCharges` in the cleaning notebook.

## Duplicate records

In [5]:
print('Full-row duplicates:', df.duplicated().sum())
print('Duplicate CustomerIDs:', df['CustomerID'].duplicated().sum())

Full-row duplicates: 25
Duplicate CustomerIDs: 25


**Observation:** A small number of duplicate CustomerID rows exist — a common artifact of pulling the same customer twice from source billing systems. These will be dropped in cleaning.

## Data quality issue: inconsistent categorical casing

In [6]:
df['Gender'].unique()

<StringArray>
['Female', 'Male', 'FEMALE', 'MALE']
Length: 4, dtype: str

**Observation:** `Gender` contains inconsistent casing (`FEMALE`/`MALE` vs `Female`/`Male`) from what looks like a secondary data source feed. Needs standardizing.

## Summary statistics (numeric columns)

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,7525.0,40.620465,13.411206,18.00,31.00,40.000,50.0000,85.00
SeniorCitizen,7525.0,0.046379,0.210318,0.00,0.00,0.000,0.0000,1.00
Tenure,7525.0,29.934086,21.207223,0.00,12.00,26.000,44.0000,72.00
MonthlyCharges,7525.0,76.259045,29.323026,18.25,64.88,84.280,98.6300,128.35
TotalCharges,7488.0,2239.247240,1899.074496,0.00,723.91,1699.525,3331.4975,9184.57
NumberOfComplaints,7525.0,0.561595,0.769976,0.00,0.00,0.000,1.0000,6.00
CustomerSatisfactionScore,7525.0,3.339402,0.785904,1.00,3.00,3.000,4.0000,5.00
SupportTickets,7525.0,1.976611,1.578526,0.00,1.00,2.000,3.0000,10.00
CustomerLifetimeValue,7525.0,3757.517553,2395.243207,25.00,1845.98,3323.800,5217.1500,13178.48


## Unique values per categorical column

In [8]:
cat_cols = ['Gender','MaritalStatus','Dependents','ContractType','InternetService','PhoneService',
            'StreamingTV','StreamingMovies','OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','PaymentMethod','PaperlessBilling','ReferralStatus']
for c in cat_cols:
    print(c, '->', df[c].unique())

Gender -> <StringArray>
['Female', 'Male', 'FEMALE', 'MALE']
Length: 4, dtype: str
MaritalStatus -> <StringArray>
['Married', 'Widowed', 'Single', 'Divorced']
Length: 4, dtype: str
Dependents -> <StringArray>
['Yes', 'No']
Length: 2, dtype: str
ContractType -> <StringArray>
['Month-to-Month', 'Two Year', 'One Year']
Length: 3, dtype: str
InternetService -> <StringArray>
['Fiber Optic', 'No', 'DSL']
Length: 3, dtype: str
PhoneService -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingTV -> <StringArray>
['No', 'No Internet Service', 'Yes']
Length: 3, dtype: str
StreamingMovies -> <StringArray>
['No', 'No Internet Service', 'Yes']
Length: 3, dtype: str
OnlineSecurity -> <StringArray>
['No', 'No Internet Service', 'Yes']
Length: 3, dtype: str
OnlineBackup -> <StringArray>
['Yes', 'No Internet Service', 'No']
Length: 3, dtype: str
DeviceProtection -> <StringArray>
['No', 'No Internet Service', 'Yes']
Length: 3, dtype: str
TechSupport -> <StringArray>
['No', 'No Internet Service'

## Memory usage

In [9]:
df.memory_usage(deep=True).sum() / 1024**2 , 'MB'

(np.float64(7.676688194274902), 'MB')

## Target variable distribution

This is the key number for the whole project — it tells us how imbalanced the classification problem is and sets our baseline.

In [10]:
counts = df['Churn'].value_counts()
pct = df['Churn'].value_counts(normalize=True) * 100
print(counts)
print(pct.round(1))

Churn
No     5133
Yes    2392
Name: count, dtype: int64
Churn
No     68.2
Yes    31.8
Name: proportion, dtype: float64


**Business takeaway:** Roughly **3 in 10 customers churn**. This is a meaningfully imbalanced but not extreme classification problem — accuracy alone would be misleading, so ROC-AUC, precision, and recall are prioritized in later evaluation. This also tells the business that ~30% of the customer base is currently walking out the door every period, which directly motivates the retention-focused analysis that follows.